In [13]:
from transformers import AutoModel
import torch

In [22]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [23]:
model_dtype = torch.bfloat16
model = AutoModel.from_pretrained("ACE-Step/acestep-v15-turbo-shift1", trust_remote_code=True,
                                  attn_implementation="eager", dtype=model_dtype).to(device)

In [24]:
text_hidden_states = torch.randn(2, 77, 1024, dtype=model_dtype, device=device)
text_attention_mask = torch.ones(2, 77, dtype=model_dtype, device=device)
lyric_hidden_states = torch.randn(2, 123, 1024, dtype=model_dtype, device=device)
lyric_attention_mask = torch.ones(2, 123, dtype=model_dtype, device=device)
refer_audio_acoustic_hidden_states_packed = torch.randn(3, 750, 64, dtype=model_dtype, device=device)
refer_audio_order_mask = torch.LongTensor([0, 0, 1]).to(device)

base_seconds = 10
frames_per_second = 25
base_seq_len = base_seconds * frames_per_second

hidden_states = torch.randn(2, base_seq_len, 64, dtype=model_dtype, device=device)
attention_mask = torch.ones(2, base_seq_len, dtype=model_dtype, device=device)

pad_start = max(base_seq_len // 2, 1)
attention_mask[0, pad_start:] = 0
chunk_mask = torch.ones(2, base_seq_len, 64, dtype=model_dtype, device=device)
chunk_mask[0, pad_start:] = 0

silence_latent = torch.randn(2, base_seq_len, 64, dtype=model_dtype, device=device)
src_latents = torch.randn(2, base_seq_len, 64, dtype=model_dtype, device=device)
is_covers = False

seconds = 10
infer_steps = 50

seq_len = seconds * frames_per_second

cur_hidden_states = torch.randn(2, seq_len, 64, dtype=model_dtype, device=device)
cur_attention_mask = torch.ones(2, seq_len, dtype=model_dtype, device=device)
cur_chunk_mask = torch.ones(2, seq_len, 64, dtype=model_dtype, device=device)
cur_silence_latent = torch.randn(2, seq_len, 64, dtype=model_dtype, device=device)
cur_src_latents = torch.randn(2, seq_len, 64, dtype=model_dtype, device=device)

In [25]:
outputs = model.generate_audio(
    text_hidden_states=text_hidden_states,
    text_attention_mask=text_attention_mask,
    lyric_hidden_states=lyric_hidden_states,
    lyric_attention_mask=lyric_attention_mask,
    refer_audio_acoustic_hidden_states_packed=refer_audio_acoustic_hidden_states_packed,
    refer_audio_order_mask=refer_audio_order_mask,
    src_latents=cur_src_latents,
    chunk_masks=cur_chunk_mask,
    silence_latent=cur_silence_latent,
    infer_steps=infer_steps,
    is_covers=is_covers,
    seed=1234,
)

RuntimeError: mat1 and mat2 must have the same dtype, but got Float and BFloat16